# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

## Plan to make the project:

Add a python code executor tool
Add a python code writer tool
Use a LLM give it the necessary tools, add system prompt
Add gradio Ui, add audio response

Checkpoint 1 - commit files

Add plot & streaming functionality

Checkpoint 2

Add voice input functionality

Submit

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "prices.db"

In [ ]:
system_message = """You are a technical AI assistant with access to two tools:

1. create_python_code → generates Python code
2. run_python_code → executes Python code

Your goal is to answer user queries accurately using reasoning and tools when appropriate.

---

### TOOL USAGE RULES (STRICT)

You must follow these rules exactly:

1. If the user asks for Python code explicitly:
   → Call create_python_code ONLY
   → Do NOT call run_python_code

2. If the user provides Python code and asks for its output:
   → Call run_python_code ONLY
   → Pass the given code directly without modification

3. If the user asks a problem that can benefit from computation 
   (math, logic, data processing, plotting, etc.):
   → FIRST call create_python_code
   → THEN pass the EXACT SAME code to run_python_code
   → DO NOT modify, edit, or reinterpret the code in any way

4. NEVER write or modify code yourself if create_python_code is used
   → The code from create_python_code must be passed AS-IS to run_python_code

---

### IMPORTANT CONSTRAINTS

- You are NOT allowed to alter tool outputs
- You must NOT regenerate or rewrite code after it is created
- You must NOT skip steps in the tool chain when both tools are needed
- Always rely on tools for computation instead of doing it manually
- Always save the plot as "plot.png" in the current directory.
- Do NOT use plt.show().

---

### RESPONSE BEHAVIOR

- Use tools whenever they improve accuracy
- Keep final answers clear and concise
- Do not mention internal tool logic unless necessary
- Assume plots or files generated will be shown to the user automatically
- Do NOT say "I cannot display images"

---

### THINKING

- Think step by step before deciding which tool(s) to use
- Choose the minimal correct sequence of actions

---

Your objective is to correctly decide:
- whether to use tools
- which tool(s) to use
- and in what order

while strictly following all rules above."""

In [ ]:
import subprocess
import tempfile
import os
import sys

def run_python_code(code: str):
    try:
        # Create a temporary file
        with tempfile.NamedTemporaryFile(delete=False, suffix=".py", mode="w") as temp_file:
            temp_file.write(code)
            temp_path = temp_file.name

        # Run the Python file
        result = subprocess.run(
            [sys.executable, temp_path],
            capture_output=True,
            text=True,
            timeout=30  # prevent infinite loops
        )

        return {
            "stdout": result.stdout,
            "stderr": result.stderr,
            "returncode": result.returncode
        }

    except subprocess.TimeoutExpired:
        return {
            "stdout": "",
            "stderr": "Execution timed out",
            "returncode": -1
        }

    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

In [ ]:
# Example usage
code = """
print("Hello World")
x = 5
print(x * 2)
"""

run_python_code(code)

In [ ]:
def create_python_code(question: str) -> str:
    MODEL = "gpt-4.1-mini"
    system_message = """You are a Python code generator.

Your task is to write complete, executable Python code that solves the user's query.

STRICT RULES:
- Output ONLY valid Python code. Do NOT include explanations.
- Do NOT use markdown formatting (no ``` or ```python).
- The code must run without errors as-is.
- Include all necessary imports.
- Do not assume any pre-defined variables.
- Always use print() to display final results.
- Always use UTF-8 compatible characters
- Avoid special unicode symbols like π, use 'pi' instead
- Save plots as "plot.png" in the current directory

FOR PLOTS:
- If the task involves plotting, use matplotlib.
- Always save the plot as "plot.png" in the current directory.
- Do NOT use plt.show().
- After saving, print exactly: plot saved at plot.png and shown to the user.

CODE QUALITY:
- Keep code clean and readable.
- Avoid unnecessary complexity.
- Ensure correctness over cleverness.

Your output must be directly executable Python code and nothing else."""

    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": question}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
(create_python_code("What is the sum of the first 10 natural numbers?"))

In [ ]:
run_python_code(create_python_code("What is the sum of the first 10 natural numbers?"))

In [ ]:
run_code_function = {
    "name": "run_python_code",
    "description": "Run python code and get the output. The input should be a string of python code. The output will be a dictionary with keys stdout, stderr and returncode.",
    "parameters": {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "The python code to run"
            }
        },
        "required": ["code"]
    }
}

create_code_function = {
    "name": "create_python_code",
    "description": "Create python code to answer a question or create a plot. The input should be a string of the question. The output will be a string of python code that can be run to get the answer to the question.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question to answer with python code"
            }
        },
        "required": ["question"]
    }
}

In [ ]:
tools = [{"type": "function", "function": run_code_function}, {"type": "function", "function": create_code_function}]

In [ ]:
def transcribe_audio(audio_path):
    with open(audio_path, "rb") as audio_file:
        transcript = openai.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",
            file=audio_file
        )
    return transcript.text

In [ ]:
def handle_voice_input(audio, history):
    if audio is None:
        return history

    text = transcribe_audio(audio)
    print(f"Transcribed: {text}")

    return history + [{"role": "user", "content": text}]

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="coral",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

voice = talker("Hello, how can I help you today?")

In [ ]:
def chat(history):
    image_generated = False
    clean_history = []

    for h in history:
        content = h["content"]
        
        # If it's a tuple (image), skip it
        if isinstance(content, tuple):
            continue
        
        # If it's a list, extract only text
        if isinstance(content, list):
            content = " ".join([c for c in content if isinstance(c, str)])
        
        clean_history.append({
            "role": h["role"],
            "content": content
        })

    messages = [{"role": "system", "content": system_message}] + clean_history

    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print(f"Initial response: {response.choices[0].message.content}, finish_reason: {response.choices[0].finish_reason}")

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"Tool call message: {message}")
        responses, image_generated = handle_tool_calls(message)
        print(f'tool call responses: {responses}')
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        print(f"Response after tool calls: {response}")

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]

    image_path = "plot.png"
    if image_generated and os.path.exists(image_path):
        history += [{
            "role": "assistant",
            "content": (image_path,)
        }]
        print(f'Image generated at {image_path}')
        # os.remove(image_path)

    # voice = talker(reply)
    
    return history, voice

In [ ]:
def handle_tool_calls(message):
    responses = []
    image_generated = False

    for tool_call in message.tool_calls:
        if tool_call.function.name == "run_python_code":
            arguments = json.loads(tool_call.function.arguments)
            code = arguments.get('code')
            output = run_python_code(code)
            responses.append({
                "role": "tool",
                "content": json.dumps(output),
                "tool_call_id": tool_call.id
            })
            if "plot.png" in output.get("stdout", ""):
                image_generated = True

        elif tool_call.function.name == "create_python_code":
            arguments = json.loads(tool_call.function.arguments)
            question = arguments.get('question')
            code = create_python_code(question)
            responses.append({
                "role": "tool",
                "content": code,
                "tool_call_id": tool_call.id
            })
    
    return responses, image_generated 

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        print(f"User message: {message}")
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=400, type="messages")

    with gr.Row():
        audio_output = gr.Audio(autoplay=True)

    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    with gr.Row():
        audio_input = gr.Audio(sources=["microphone"], type="filepath")

    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    ).then(
        chat,
        inputs=chatbot,
        outputs=[chatbot, audio_output]
    )

    audio_input.change(
        handle_voice_input,
        inputs=[audio_input, chatbot],
        outputs=chatbot
    ).then(
        chat,
        inputs=chatbot,
        outputs=[chatbot, audio_output]
    )
ui.launch(inbrowser=True)